## Judilibre scraping setup

This step downloads first-instance court decisions from the Cassation **Judilibre** API and stores them locally for the rest of the pipeline.

- Imports the required libraries (`requests`, `os`, `tqdm`, `pandas`, ...)
- Defines the API authentication header (`KeyId`, read from the `JUDILIBRE_API_KEY` environment variable)
- Creates the local `artifacts/raw_decisions/` folder for downloaded decisions

In [ ]:
import json
import os
import re
import time
from datetime import timedelta, date

import requests
import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
my_headers = {'KeyId' : os.environ["JUDILIBRE_API_KEY"]}

In [ ]:
# Create directory if it does not exist
if not os.path.exists('artifacts/raw_decisions'):
    os.makedirs('artifacts/raw_decisions')  # not shipped — regenerated by this step (see DATA.md)

## Generating INSEE codes for judicial courts

- **Raw list**: contains all labels of judicial courts (tribunaux judiciaires) in France.
- **Cleaning**: a `clean_city()` function extracts the municipality name from the label (removing "Tribunal judiciaire de...").
- **API query**: the `get_insee_code()` function uses the public `geo.api.gouv.fr` API to retrieve each municipality's **INSEE code**.
- **Loop**: iterates over all courts and exports results to an Excel file.

In [ ]:
# ----------------------------------------------------------
# 1) Raw list of judicial courts
# ----------------------------------------------------------
tribunaux_raw = [
    "Tribunal judiciaire de Bourg-en-Bresse",
    "Tribunal judiciaire de Laon",
    "Tribunal judiciaire de Soissons",
    "Tribunal judiciaire de Saint-Quentin",
    "Tribunal judiciaire de Cusset",
    "Tribunal judiciaire de Moulins",
    "Tribunal judiciaire de Montluçon",
    "Tribunal judiciaire de Digne-les-Bains",
    "Tribunal judiciaire de Gap",
    "Tribunal judiciaire de Grasse",
    "Tribunal judiciaire de Nice",
    "Tribunal judiciaire de Privas",
    "Tribunal judiciaire de Charleville-Mézières",
    "Tribunal judiciaire de Foix",
    "Tribunal judiciaire de Troyes",
    "Tribunal judiciaire de Carcassonne",
    "Tribunal judiciaire de Narbonne",
    "Tribunal judiciaire de Rodez",
    "Tribunal judiciaire d'Aix-en-Provence",
    "Tribunal judiciaire de Marseille",
    "Tribunal judiciaire de Tarascon",
    "Tribunal judiciaire de Lisieux",
    "Tribunal judiciaire de Caen",
    "Tribunal judiciaire d'Aurillac",
    "Tribunal judiciaire d'Angoulême",
    "Tribunal judiciaire de Saintes",
    "Tribunal judiciaire de La Rochelle",
    "Tribunal judiciaire de Bourges",
    "Tribunal judiciaire de Tulle",
    "Tribunal judiciaire de Brive-la-Gaillarde",
    "Tribunal judiciaire d'Ajaccio",
    "Tribunal judiciaire de Bastia",
    "Tribunal judiciaire de Dijon",
    "Tribunal judiciaire de Saint-Brieuc",
    "Tribunal judiciaire de Saint-Malo",
    "Tribunal judiciaire de Guéret",
    "Tribunal judiciaire de Périgueux",
    "Tribunal judiciaire de Bergerac",
    "Tribunal judiciaire de Besançon",
    "Tribunal judiciaire de Montbéliard",
    "Tribunal judiciaire de Valence",
    "Tribunal judiciaire d'Evreux",
    "Tribunal judiciaire de Chartres",
    "Tribunal judiciaire de Quimper",
    "Tribunal judiciaire de Brest",
    "Tribunal judiciaire de Nîmes",
    "Tribunal judiciaire d'Alès",
    "Tribunal judiciaire de St Gaudens",
    "Tribunal judiciaire de Toulouse",
    "Tribunal judiciaire d'Auch",
    "Tribunal judiciaire de Libourne",
    "Tribunal judiciaire de Bordeaux",
    "Tribunal judiciaire de Béziers",
    "Tribunal judiciaire de Montpellier",
    "Tribunal judiciaire de Rennes",
    "Tribunal judiciaire de Châteauroux",
    "Tribunal judiciaire de Tours",
    "Tribunal judiciaire de Bourgoin-Jallieu",
    "Tribunal judiciaire de Grenoble",
    "Tribunal judiciaire de Vienne",
    "Tribunal judiciaire de Lons-le-Saunier",
    "Tribunal judiciaire de Mont-de-Marsan",
    "Tribunal judiciaire de Dax",
    "Tribunal judiciaire de Blois",
    "Tribunal judiciaire de Saint-Etienne",
    "Tribunal judiciaire de Roanne",
    "Tribunal judiciaire du Puy-en-Velay",
    "Tribunal judiciaire de Nantes",
    "Tribunal judiciaire de Saint-Nazaire",
    "Tribunal judiciaire de Montargis",
    "Tribunal judiciaire d'Orléans",
    "Tribunal judiciaire de Cahors",
    "Tribunal judiciaire d'Agen",
    "Tribunal judiciaire de Mende",
    "Tribunal judiciaire de Saumur",
    "Tribunal judiciaire d'Angers",
    "Tribunal judiciaire de Coutances",
    "Tribunal judiciaire de Cherbourg-en-Cotentin",
    "Tribunal judiciaire de Châlons-en-Champagne",
    "Tribunal judiciaire de Reims",
    "Tribunal judiciaire de Chaumont",
    "Tribunal judiciaire de Laval",
    "Tribunal judiciaire de Nancy",
    "Tribunal judiciaire de Val de Briey",
    "Tribunal judiciaire de Bar-le-Duc",
    "Tribunal judiciaire de Verdun",
    "Tribunal judiciaire de Vannes",
    "Tribunal judiciaire de Lorient",
    "Tribunal judiciaire de Thionville",
    "Tribunal judiciaire de Metz",
    "Tribunal judiciaire de Sarreguemines",
    "Tribunal judiciaire de Nevers",
    "Tribunal judiciaire de Cambrai",
    "Tribunal judiciaire de Valenciennes",
    "Tribunal judiciaire d'Avesnes-sur-Helpe",
    "Tribunal judiciaire de Douai",
    "Tribunal judiciaire de Lille",
    "Tribunal judiciaire de Dunkerque",
    "Tribunal judiciaire de Beauvais",
    "Tribunal judiciaire de Senlis",
    "Tribunal judiciaire de Compiègne",
    "Tribunal judiciaire d'Alençon",
    "Tribunal judiciaire d'Argentan",
    "Tribunal judiciaire d'Arras",
    "Tribunal judiciaire de Saint-Omer",
    "Tribunal judiciaire de Boulogne-sur-Mer",
    "Tribunal judiciaire de Béthune",
    "Tribunal judiciaire de Clermont-Ferrand",
    "Tribunal judiciaire de Pau",
    "Tribunal judiciaire de Bayonne",
    "Tribunal judiciaire de Tarbes",
    "Tribunal judiciaire de Perpignan",
    "Tribunal judiciaire de Strasbourg",
    "Tribunal judiciaire de Saverne",
    "Tribunal judiciaire de Colmar",
    "Tribunal judiciaire de Mulhouse",
    "Tribunal judiciaire de Villefranche-sur-Saône",
    "Tribunal judiciaire de Lyon",
    "Tribunal judiciaire de Vesoul",
    "Tribunal judiciaire de Chalon-sur-Saône",
    "Tribunal judiciaire de Mâcon",
    "Tribunal judiciaire du Mans",
    "Tribunal judiciaire de Chambéry",
    "Tribunal judiciaire d'Albertville",
    "Tribunal judiciaire de Thonon-les-Bains",
    "Tribunal judiciaire d'Annecy",
    "Tribunal judiciaire de Bonneville",
    "Tribunal judiciaire de Paris",
    "Tribunal judiciaire de Rouen",
    "Tribunal judiciaire du Havre",
    "Tribunal judiciaire de Dieppe",
    "Tribunal judiciaire de Fontainebleau",
    "Tribunal judiciaire de Meaux",
    "Tribunal judiciaire de Melun",
    "Tribunal judiciaire de Versailles",
    "Tribunal judiciaire de Niort",
    "Tribunal judiciaire d'Amiens",
    "Tribunal judiciaire de Castres",
    "Tribunal judiciaire d'Albi",
    "Tribunal judiciaire de Montauban",
    "Tribunal judiciaire de Draguignan",
    "Tribunal judiciaire de Toulon",
    "Tribunal judiciaire de Carpentras",
    "Tribunal judiciaire d'Avignon",
    "Tribunal judiciaire des Sables-d'Olonne",
    "Tribunal judiciaire de La Roche-sur-Yon",
    "Tribunal judiciaire de Poitiers",
    "Tribunal judiciaire de Limoges",
    "Tribunal judiciaire d'Epinal",
    "Tribunal judiciaire d'Auxerre",
    "Tribunal judiciaire de Sens",
    "Tribunal judiciaire de Belfort",
    "Tribunal judiciaire d'Évry-Courcouronnes",
    "Tribunal judiciaire de Nanterre",
    "Tribunal judiciaire de Bobigny",
    "Tribunal judiciaire de Créteil",
    "Tribunal judiciaire de Pontoise",
    "Tribunal judiciaire de Pointe-à-Pitre",
    "Tribunal judiciaire de Basse-Terre",
    "Tribunal judiciaire de Fort-de-France",
    "Tribunal judiciaire de Cayenne",
    "Tribunal judiciaire de Saint-Pierre",
    "Tribunal judiciaire de Saint-Denis-de-La-Réunion",
    "Tribunal de Première Instance de Saint-Pierre-et-Miquelon",
    "Tribunal judiciaire de Mamoudzou",
    "Tribunal de Première Instance de Mata-Utu",
    "Tribunal de Première Instance de Papeete",
    "Tribunal de Première Instance de Nouméa"
]

# ----------------------------------------------------------
# 2) Helper to extract city names from court labels
# ----------------------------------------------------------
import re

def clean_city(label: str) -> str:
    # Remove the prefix "Tribunal judiciaire de / d'/ du ..."
    city = re.sub(r"^Tribunal judiciaire (du|de|d'|d')\s*", "", label, flags=re.I)
    return city.strip()

# ----------------------------------------------------------
# 3) Query the official geo.api.gouv.fr API
# (INSEE database) to retrieve municipality codes.
# ----------------------------------------------------------
import requests, pandas as pd, time

def get_insee_code(city: str) -> str:
    """Return the 5-character INSEE code (or 2A/2B + 3 digits)."""
    url = (
        "https://geo.api.gouv.fr/communes"
        f"?nom={requests.utils.quote(city)}"
        "&fields=code&limit=1"
    )
    resp = requests.get(url, timeout=10).json()
    if not resp:
        raise ValueError(f"Unknown city: {city}")
    return resp[0]["code"]

rows = []
for trib in tribunaux_raw:
    ville = clean_city(trib)
    try:
        code = get_insee_code(ville)
        rows.append({"Tribunal": trib, "Code": f"tj{code}"})
    except Exception as e:
        rows.append({"Tribunal": trib, "Code": f" {e}"})
    # Rate-limit courtesy delay for the public API
    time.sleep(0.15)

df = pd.DataFrame(rows)
# Export to Excel
os.makedirs("artifacts/reference", exist_ok=True)  # not shipped — regenerated by this step (see DATA.md)
df.to_excel("artifacts/reference/tribunal_insee_codes_draft.xlsx", index=False)

## Extracting INSEE codes from an Excel file

This script loads an Excel file containing courts and their codes, then extracts the list of valid codes:

- `file_path`: path to the `code_tribunaux_insee.xlsx` file.
- `pd.read_excel(...)`: reads the Excel file into a DataFrame.
- `df.iloc[:, 1]`: selects the 2nd column (TJ codes), dropping missing values.
- Filtering: keeps only entries starting with `tj` (valid codes).

In [ ]:
import pandas as pd

file_path = "DATA/inputs/tribunal_insee_codes.xlsx"
df = pd.read_excel(file_path)

# Extract codes from the second column
code_list = df.iloc[:, 1].dropna().tolist()

# Filter out invalid codes
code_list = [code for code in code_list if code.startswith("tj")]

len(code_list)

## Detecting active TJs (decisions published since 2020)

This script queries the Judilibre API to determine which judicial courts (`tjXXXXX`) have **at least one decision published since January 1, 2020**.

### Steps:

- Authentication via a `KeyId` header.
- Global period: from `2020-01-01` to today.
- For each TJ in `TOUS_TJ`:
  - An API request is sent with `jurisdiction=tj`, `location=<code_tj>`, and the date range.
  - If at least one result is returned, the TJ is marked as active.
- Displays the final list of active TJs.

In [ ]:
import requests
from datetime import date

# Authentication
my_headers = {'KeyId': os.environ["JUDILIBRE_API_KEY"]}

# List of TJs to test
TOUS_TJ = [
'tj02408', 'tj02722', 'tj02691', 'tj03095', 'tj03190', 'tj03185', 'tj04070', 'tj05061', 'tj06069', 'tj06088', 'tj07186', 'tj08105', 'tj09122', 'tj10387', 'tj11069', 'tj11262', 'tj12202', 'tj13001', 'tj13055', 'tj13108', 'tj14366', 'tj14118', 'tj15014', 'tj16015', 'tj17300', 'tj18033', 'tj19272', 'tj19031', 'tj2A004', 'tj2B033', 'tj21231', 'tj22278', 'tj35288', 'tj23096', 'tj24322', 'tj24037', 'tj25056', 'tj25388', 'tj26362', 'tj27229', 'tj28085', 'tj29232', 'tj29019', 'tj30189', 'tj30007', 'tj31483', 'tj31555', 'tj32013', 'tj33243', 'tj33063', 'tj34032', 'tj34172', 'tj35238', 'tj36044', 'tj37261', 'tj38053', 'tj38185', 'tj38544', 'tj39300', 'tj40192', 'tj40088', 'tj41018', 'tj42218', 'tj42187', 'tj43157', 'tj44109', 'tj44184', 'tj45208', 'tj45234', 'tj46042', 'tj47001', 'tj48095', 'tj49328', 'tj49007', 'tj50147', 'tj50129', 'tj51108', 'tj51454', 'tj53130', 'tj54395', 'tj54099', 'tj55029', 'tj55545', 'tj56260', 'tj56121', 'tj57672', 'tj57463', 'tj57631', 'tj58194', 'tj59122', 'tj59606', 'tj59036', 'tj59178', 'tj59350', 'tj59183', 'tj60057', 'tj60612', 'tj60159', 'tj61001', 'tj61006', 'tj62041', 'tj62765', 'tj62160', 'tj62119', 'tj63113', 'tj64445', 'tj64102', 'tj65440', 'tj66136', 'tj67482', 'tj67437', 'tj68066', 'tj68224', 'tj69264', 'tj69123', 'tj70550', 'tj71076', 'tj71270', 'tj72181', 'tj73065', 'tj73011', 'tj74281', 'tj74010', 'tj74042', 'tj75056', 'tj76540', 'tj76351', 'tj76217', 'tj77186', 'tj77284', 'tj77288', 'tj78646', 'tj79191', 'tj80021', 'tj81065', 'tj81004', 'tj82121', 'tj83050', 'tj83137', 'tj84031', 'tj84007', 'tj85194', 'tj85191', 'tj86194', 'tj87085', 'tj88160', 'tj89024', 'tj89387', 'tj90010', 'tj91228', 'tj92050', 'tj93008', 'tj94028', 'tj95500', 'tj97120', 'tj97105', 'tj97209', 'tj97302', 'tj97416', 'tj97411', 'tj97611', 'tj98613', 'tj98735', 'tj98818'
]

# Global period: from 2020-01-01 to today
date_start = "2020-01-01"
date_end = date.today().isoformat()

# Active TJs (having at least one decision since 2020)
tjs_actifs = []

# Test each TJ
for tj in TOUS_TJ:
    try:
        response = requests.get(
            "https://sandbox-api.piste.gouv.fr/cassation/judilibre/v1.0/export",
            params={
                "jurisdiction": "tj",
                "location": tj,
                "date_start": date_start,
                "date_end": date_end,
                "batch_size": 1,
                "batch": 0
            },
            headers=my_headers,
            timeout=10
        )
        response.raise_for_status()
        results = response.json().get("results", [])
        nb = len(results)
        if nb > 0:
            print(f"  {tj}: {nb} decision(s) found")
            tjs_actifs.append(tj)
        else:
            print(f"  {tj}: no decisions found")
    except Exception as e:
        print(f"  Error for {tj} -> {e}")

# Final summary
print(f"\n{len(tjs_actifs)} active TJs found since 2020:")
print(tjs_actifs)

## Judilibre scraping -- Downloading TJ decisions since 2020

This script retrieves all decisions published since 2020 for a list of **active judicial courts (`tjs_actifs`)** via the Judilibre sandbox API.

### Authentication
Uses a `KeyId` header for API access.

### Storage structure

- Root directory: `raw_decisions/`
- One **subfolder per TJ** is created automatically (`tj75056/`, `tj13055/`, etc.)
- Each decision is saved as a separate JSON file named by its unique identifier.

### Download logic

- Time is traversed backwards month by month (30-day windows).
- If a window returns more than 1000 decisions, it is subdivided into 5-day windows to stay within the API batch limit.

In [ ]:
import os
import json
import requests
from datetime import date, timedelta

# API authentication
my_headers = {'KeyId': os.environ["JUDILIBRE_API_KEY"]}

# Active TJs
tjs_actifs = ['tj02722', 'tj02691', 'tj03185', 'tj06069', 'tj06088', 'tj09122', 'tj12202', 'tj13001', 'tj13055', 'tj14366', 'tj14118', 'tj15014', 'tj16015', 'tj21231', 'tj22278', 'tj35288', 'tj24322', 'tj24037', 'tj26362', 'tj27229', 'tj28085', 'tj29232', 'tj30189', 'tj31483', 'tj31555', 'tj33063', 'tj34172', 'tj35238', 'tj37261', 'tj38053', 'tj38185', 'tj38544', 'tj42218', 'tj44109', 'tj45208', 'tj45234', 'tj48095', 'tj49328', 'tj49007', 'tj50147', 'tj50129', 'tj53130', 'tj54395', 'tj54099', 'tj56260', 'tj56121', 'tj57672', 'tj57463', 'tj57631', 'tj59122', 'tj59606', 'tj59350', 'tj60159', 'tj61001', 'tj62041', 'tj62160', 'tj62119', 'tj63113', 'tj66136', 'tj67482', 'tj68066', 'tj68224', 'tj69123', 'tj72181', 'tj75056', 'tj76351', 'tj77284', 'tj78646', 'tj79191', 'tj80021', 'tj82121', 'tj83050', 'tj83137', 'tj84031', 'tj84007', 'tj85194', 'tj85191', 'tj86194', 'tj89024', 'tj91228', 'tj92050', 'tj93008', 'tj94028', 'tj95500', 'tj97411', 'tj98818']

# Root directory
root_dir = "artifacts/raw_decisions"  # not shipped — regenerated by this step (see DATA.md)
os.makedirs(root_dir, exist_ok=True)

# Save decisions to a TJ-specific folder
def save_decisions(results, tj_dir):
    for item in results:
        identifiant = item["id"]
        filepath = os.path.join(tj_dir, f"{identifiant}.json")
        with open(filepath, "w", encoding="utf8") as f:
            json.dump(item, f, ensure_ascii=False)

# Iterate over TJs
for tj in tjs_actifs:
    print(f"\n=== Processing {tj} ===")
    tj_dir = os.path.join(root_dir, tj)
    os.makedirs(tj_dir, exist_ok=True)

    # Main loop: go backwards in time month by month
    date_end = date.today()
    delta_month = timedelta(days=30)
    date_start = date_end - delta_month

    while date_start.year > 2020:
        date_start_iso = date_start.isoformat()
        date_end_iso = date_end.isoformat()

        try:
            response = requests.get(
                "https://sandbox-api.piste.gouv.fr/cassation/judilibre/v1.0/export",
                params={
                    "jurisdiction": "tj",
                    "location": tj,
                    "date_start": date_start_iso,
                    "date_end": date_end_iso,
                    "batch_size": 1000,
                    "batch": 0
                },
                headers=my_headers,
                timeout=15
            )
            response.raise_for_status()
            results = response.json().get("results", [])
            nb = len(results)
            print(f"  {date_start_iso} -> {date_end_iso}: {nb} decisions")

            if nb < 1000:
                save_decisions(results, tj_dir)

            else:
                print(f"    Over 1000 decisions -> splitting into 5-day windows")
                inner_end = date_end
                delta_5d = timedelta(days=5)
                inner_start = inner_end - delta_5d

                while inner_start >= date_start:
                    inner_start_iso = inner_start.isoformat()
                    inner_end_iso = inner_end.isoformat()

                    try:
                        response_sub = requests.get(
                            "https://sandbox-api.piste.gouv.fr/cassation/judilibre/v1.0/export",
                            params={
                                "jurisdiction": "tj",
                                "location": tj,
                                "date_start": inner_start_iso,
                                "date_end": inner_end_iso,
                                "batch_size": 1000,
                                "batch": 0
                            },
                            headers=my_headers,
                            timeout=15
                        )
                        response_sub.raise_for_status()
                        sub_results = response_sub.json().get("results", [])
                        print(f"      {inner_start_iso} -> {inner_end_iso}: {len(sub_results)} decisions")
                        save_decisions(sub_results, tj_dir)

                    except requests.exceptions.RequestException as e:
                        print(f"      Error 5d window: {inner_start_iso} -> {inner_end_iso}: {e}")

                    inner_end = inner_start
                    inner_start = inner_end - delta_5d

            # Move backwards in time
            date_end = date_start
            date_start = date_end - delta_month

        except requests.exceptions.RequestException as e:
            print(f"  Error for {tj} [{date_start_iso} -> {date_end_iso}]: {e}")
            # Move backwards even on error to avoid getting stuck
            date_end = date_start
            date_start = date_end - delta_month